# Figure 1f/1g — SM-102 LNP-treated spleen neighborhoods

This publication notebook is a cleaned duplicate of
`20260614_SpatialOmics_Neighborhoods_codebook_matched_Yining.ipynb`.

It displays only:

- **Figure 1f:** cell-type composition of the treated-spleen spatial
  neighborhoods; and
- **Figure 1g:** abundance of those neighborhoods among full-barcode
  codebook-matched LNP+ cells versus LNP− cells.

The analysis is restricted to `reg001` (SM-102 LNP-treated spleen). Whole-tissue
multi-region exploration, liver, luciferase/transfection classes, spatial
overlays, and unrelated plots were removed. This notebook is distributed
unexecuted and does not write figures or tables.


## 1. Load the frozen treated-spleen cell table

The compact input contains one row per segmented cell with its centroid,
full-barcode LNP call, and previously established cell-type annotation.

Cell segmentation, clustering, and cell-type annotation were performed with the
previously published workflow and are frozen upstream inputs. Cite the
corresponding manuscript reference; those upstream notebooks are intentionally
not included here.


In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors

try:
    from IPython.display import display
except ImportError:
    display = print

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "Liberation Sans"],
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 8,
    "xtick.labelsize": 6.3,
    "ytick.labelsize": 6.7,
    "figure.dpi": 160,
})


def resolve_data_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for candidate in (
            base / "Manuscripts" / "NanoSTAMP" / "Data" / "Figure_1f_1g_Spleen_Neighborhoods",
            base / "Data" / "Figure_1f_1g_Spleen_Neighborhoods",
        ):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(
        "Could not locate Data/Figure_1f_1g_Spleen_Neighborhoods"
    )


DATA_ROOT = resolve_data_root()
cells = pd.read_csv(
    DATA_ROOT / "Precomputed_Analysis_Input" / "reg001_neighborhood_cells.csv.gz",
    dtype={"region": str, "cell_type": str},
)
cells["cell"] = pd.to_numeric(cells["cell"], errors="raise").astype("int64")
cells["x"] = pd.to_numeric(cells["x"], errors="raise")
cells["y"] = pd.to_numeric(cells["y"], errors="raise")
cells["lnp_positive"] = cells["lnp_positive"].astype(bool)
cells["lnp_negative"] = ~cells["lnp_positive"]
cells["Cell Type"] = cells["cell_type"].astype(str)
cells["unique_region"] = cells["region"]

if not cells["region"].eq("reg001").all():
    raise ValueError("Figure 1f/1g expects reg001 treated-spleen cells only")
if cells.duplicated(["region", "cell"]).any():
    raise ValueError("Duplicate region/cell identifiers")

print(f"reg001 cells used: {len(cells):,}")
print(f"full-barcode LNP+ cells: {cells['lnp_positive'].sum():,}")
print(f"cells with Unknown annotation: {(cells['cell_type'] == 'Unknown').sum():,}")
display(cells.head())


## 2. Neighborhood calculation

For each cell, count cell types in its 10-nearest-cell window (including the
index cell, matching the source workflow). Cluster those windows into eight
neighborhoods using `MiniBatchKMeans` with `random_state=0` and `n_init=10`.
Name each neighborhood by its dominant cell type. If multiple clusters receive
the same name, collapse them using their cell counts as weights.


In [ ]:
def get_windows(job, n_neighbors, experiments, tissue_group, x_col, y_col):
    start_time, index, tissue_name, indices = job
    job_start = time.time()
    print(f"Starting {index + 1}/{len(experiments)}: {experiments[index]}")
    tissue = tissue_group.get_group(tissue_name)
    to_fit = tissue.loc[indices, [x_col, y_col]].to_numpy()
    fit = NearestNeighbors(n_neighbors=n_neighbors).fit(
        tissue[[x_col, y_col]].to_numpy()
    )
    distances, neighbor_indices = fit.kneighbors(to_fit)
    order = distances.argsort(axis=1)
    offsets = np.arange(neighbor_indices.shape[0]) * neighbor_indices.shape[1]
    sorted_indices = neighbor_indices.ravel()[order + offsets[:, None]]
    neighbors = tissue.index.to_numpy()[sorted_indices]
    print(
        f"Finished {index + 1}/{len(experiments)}: {experiments[index]} "
        f"in {time.time() - job_start:.1f} s "
        f"({time.time() - start_time:.1f} s total)"
    )
    return neighbors.astype(np.int32)


def build_region_windows(
    cells_df,
    cluster_col="Cell Type",
    k=10,
    x_col="x",
    y_col="y",
    region_col="unique_region",
):
    cells_df = cells_df.reset_index(drop=True).copy()
    cells_df[cluster_col] = cells_df[cluster_col].astype(str).str.strip()
    dummies = pd.get_dummies(cells_df[cluster_col], dtype=np.int8)
    metadata = cells_df[
        [x_col, y_col, region_col, cluster_col, "lnp_positive", "lnp_negative"]
    ].copy()
    cells_df = pd.concat([metadata, dummies], axis=1)
    cell_types = dummies.columns.to_list()
    values = dummies.to_numpy(dtype=np.float32)

    tissue_group = cells_df.groupby(region_col)
    experiments = list(cells_df[region_col].astype(str).unique())
    jobs = [
        (
            time.time(),
            index,
            experiment,
            tissue_group.get_group(experiment).index.to_numpy(),
        )
        for index, experiment in enumerate(experiments)
    ]
    tissues = [
        get_windows(
            job,
            n_neighbors=k,
            experiments=experiments,
            tissue_group=tissue_group,
            x_col=x_col,
            y_col=y_col,
        )
        for job in jobs
    ]

    window_values = np.zeros(
        (len(cells_df), len(cell_types)),
        dtype=np.float32,
    )
    for neighbors in tissues:
        index_cells = neighbors[:, 0]
        window_values[index_cells] = values[neighbors[:, :k]].sum(axis=1)
    windows = pd.DataFrame(
        window_values,
        columns=cell_types,
        index=cells_df.index,
    )
    return cells_df, cell_types, windows


def cluster_neighborhoods(
    cells_df,
    windows,
    cell_types,
    n_neighborhoods=8,
    random_state=0,
):
    model = MiniBatchKMeans(
        n_clusters=n_neighborhoods,
        random_state=random_state,
        n_init=10,
    )
    labels = model.fit_predict(windows[cell_types].to_numpy())
    clustered = cells_df.copy()
    clustered["neighborhood"] = labels

    centroids = pd.DataFrame(
        model.cluster_centers_,
        columns=cell_types,
        index=pd.Index(range(n_neighborhoods), name="neighborhood"),
    )
    percent = centroids.div(centroids.sum(axis=1), axis=0).fillna(0) * 100
    baseline = windows[cell_types].to_numpy().mean(axis=0)
    fold_change = np.log2(
        (
            (centroids.to_numpy() + baseline)
            / (centroids.to_numpy() + baseline).sum(axis=1, keepdims=True)
        )
        / baseline
    )
    fold_change = pd.DataFrame(
        fold_change,
        index=centroids.index,
        columns=cell_types,
    )
    summary = (
        clustered.groupby("neighborhood", observed=False)
        .agg(n_cells=("neighborhood", "size"))
        .sort_values("n_cells", ascending=False)
    )
    return clustered, percent, fold_change, summary


def collapse_neighborhood_rows(table, label_map, weights):
    labeled = table.copy()
    labeled.index = pd.Index(
        [label_map.get(index, index) for index in labeled.index],
        name=table.index.name,
    )
    weights = pd.Series(weights).reindex(table.index).astype(float).fillna(0)
    weighted = labeled.mul(weights.to_numpy(), axis=0)
    grouped_values = weighted.groupby(level=0, sort=False).sum()
    grouped_weights = pd.Series(
        weights.to_numpy(),
        index=labeled.index,
    ).groupby(level=0, sort=False).sum()
    return grouped_values.div(grouped_weights, axis=0).fillna(0)


In [ ]:
K_NEIGHBORS = 10
N_NEIGHBORHOODS = 8

neighborhood_cells, cell_types, windows = build_region_windows(
    cells,
    cluster_col="Cell Type",
    k=K_NEIGHBORS,
)
clustered_cells, neighborhood_percent, neighborhood_fc, neighborhood_summary = (
    cluster_neighborhoods(
        neighborhood_cells,
        windows,
        cell_types,
        n_neighborhoods=N_NEIGHBORHOODS,
        random_state=0,
    )
)

dominant_types = neighborhood_percent.idxmax(axis=1)
label_map = {
    index: f"{cell_type} enriched"
    for index, cell_type in dominant_types.items()
}
weights = neighborhood_summary["n_cells"]

named_fc = collapse_neighborhood_rows(
    neighborhood_fc,
    label_map,
    weights,
)

# Match the source plot ordering: cluster the log2 fold-change table, then
# apply the resulting row and column order to the composition table.
ordering_grid = sns.clustermap(
    named_fc,
    vmin=-3,
    vmax=3,
    cmap="bwr",
    figsize=(4, 3),
)
row_order = ordering_grid.dendrogram_row.reordered_ind
column_order = ordering_grid.dendrogram_col.reordered_ind
plt.close(ordering_grid.fig)

named_percent = collapse_neighborhood_rows(
    neighborhood_percent,
    label_map,
    weights,
).iloc[row_order, column_order]

clustered_cells["neighborhood_name"] = (
    clustered_cells["neighborhood"].map(label_map)
)

display(neighborhood_summary)
display(named_percent)


## 3. Figure 1f — neighborhood cell-type composition

Color shows the percentage of each cell type in each named neighborhood,
capped at 30% for display.


In [ ]:
plot_df = named_percent.copy()

fig = plt.figure(figsize=(5.35, 3.75))
grid = fig.add_gridspec(
    nrows=2,
    ncols=1,
    height_ratios=[0.13, 1.0],
    hspace=0.28,
)
colorbar_ax = fig.add_subplot(grid[0, 0])
ax = fig.add_subplot(grid[1, 0])

heatmap = sns.heatmap(
    plot_df,
    cmap="Reds",
    vmin=0,
    vmax=30,
    linewidths=0,
    cbar=True,
    cbar_ax=colorbar_ax,
    cbar_kws={"orientation": "horizontal"},
    ax=ax,
)

ax.set_title(
    "SM-102 LNP spleen neighborhood composition (%)",
    loc="left",
    pad=5,
    fontweight="bold",
    fontsize=8.4,
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="both", length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center", va="top")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, va="center")
for spine in ax.spines.values():
    spine.set_visible(False)

colorbar = heatmap.collections[0].colorbar
colorbar.outline.set_visible(False)
colorbar_ax.xaxis.set_ticks_position("top")
colorbar_ax.xaxis.set_label_position("top")
colorbar_ax.tick_params(axis="x", length=0, labelsize=6.4, pad=1)
colorbar.set_label("Cell-type % (capped at 30)", labelpad=3, fontsize=6.8)
for spine in colorbar_ax.spines.values():
    spine.set_visible(False)

fig.subplots_adjust(left=0.33, right=0.98, top=0.86, bottom=0.39)
plt.show()


## 4. Figure 1g — neighborhood abundance in LNP− and LNP+ cells

For each group, percentages sum to 100% across neighborhoods. Delta labels show
the LNP+ percentage minus the LNP− percentage.


In [ ]:
comparison_frames = []
for label, mask in [
    ("Decoded LNP-", clustered_cells["lnp_negative"]),
    ("Decoded LNP+", clustered_cells["lnp_positive"]),
]:
    part = clustered_cells.loc[mask, ["neighborhood_name"]].copy()
    part["Comparison group"] = label
    comparison_frames.append(part)

comparison = pd.concat(comparison_frames, ignore_index=True)
comparison["Comparison group"] = pd.Categorical(
    comparison["Comparison group"],
    categories=["Decoded LNP-", "Decoded LNP+"],
    ordered=True,
)
abundance = (
    pd.crosstab(
        comparison["neighborhood_name"],
        comparison["Comparison group"],
        normalize="columns",
    )
    * 100
)
abundance = abundance.reindex(
    columns=["Decoded LNP-", "Decoded LNP+"]
).fillna(0)
abundance["Difference"] = (
    abundance["Decoded LNP+"] - abundance["Decoded LNP-"]
)
abundance = abundance.sort_values("Difference", ascending=True)
display(abundance)

fig, ax = plt.subplots(figsize=(4.35, 2.75))
y_positions = np.arange(len(abundance))

for index, (_, row) in enumerate(abundance.iterrows()):
    ax.plot(
        [row["Decoded LNP-"], row["Decoded LNP+"]],
        [index, index],
        color="#D0D0D0" if row["Difference"] >= 0 else "#B8B8B8",
        linewidth=1.0,
        zorder=1,
    )

ax.scatter(
    abundance["Decoded LNP-"],
    y_positions,
    s=28,
    color="#8E8E8E",
    edgecolor="white",
    linewidth=0.45,
    zorder=3,
    label="LNP- cells",
)
ax.scatter(
    abundance["Decoded LNP+"],
    y_positions,
    s=30,
    color="#D84A3A",
    edgecolor="white",
    linewidth=0.45,
    zorder=4,
    label="LNP+ cells",
)

for index, (_, row) in enumerate(abundance.iterrows()):
    ax.text(
        max(row["Decoded LNP-"], row["Decoded LNP+"]) + 0.55,
        index,
        f"{row['Difference']:+.1f}",
        ha="left",
        va="center",
        fontsize=5.8,
        color="#B12A1C" if row["Difference"] > 0 else "#555555",
    )

ax.axvline(0, color="#222222", linewidth=0.65)
ax.set_yticks(y_positions)
ax.set_yticklabels(abundance.index)
ax.set_xlabel("Cells in neighborhood (%)", labelpad=3)
ax.set_ylabel("")
ax.set_title(
    "Neighborhood abundance in SM-102 LNP-treated spleen",
    loc="left",
    pad=5,
    fontweight="bold",
    fontsize=8.0,
)
ax.text(
    0,
    -0.18,
    "Delta labels show LNP+ - LNP-",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=5.6,
    color="#555555",
    clip_on=False,
)
ax.grid(axis="x", color="#E5E5E5", linewidth=0.55)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="both", length=2.4, width=0.7, pad=2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 0.98),
    handletextpad=0.35,
    borderaxespad=0,
    fontsize=6.2,
)
ax.set_xlim(
    0,
    max(
        abundance[["Decoded LNP-", "Decoded LNP+"]].max().max() + 4.2,
        35,
    ),
)
fig.subplots_adjust(left=0.34, right=0.78, top=0.86, bottom=0.26)
plt.show()
